## Example 1 — installed version and the libsoxr it reports

In [ ]:
import importlib.metadata
import numpy as np
import soxr

assert soxr.__version__ == importlib.metadata.version("soxr"), soxr.__version__
soxr.resample(np.zeros(64, dtype=np.float32), 2, 1)
print("soxr", soxr.__version__, "on", soxr.__libsoxr_version__)

## Example 2 — 44.1 kHz to 16 kHz: a 1 kHz tone comes back as the same tone (vLLM's call)

In [ ]:
import numpy as np
import soxr

in_rate, out_rate, tone = 44100, 16000, 1000.0
for dtype in (np.float32, np.float64):
    signal = np.sin(2 * np.pi * tone * np.arange(in_rate) / in_rate).astype(dtype)
    resampled = soxr.resample(signal, in_rate, out_rate)  # vllm/multimodal/audio.py, resample_audio_soxr
    assert resampled.dtype == dtype and resampled.shape == (out_rate,), (resampled.dtype, resampled.shape)
    expected = np.sin(2 * np.pi * tone * np.arange(out_rate) / out_rate)
    # The first and last few hundred samples carry the filter's edge effect; the middle must match the analytic tone.
    error = float(np.max(np.abs(resampled[1000:-1000] - expected[1000:-1000])))
    assert error < 1e-4, (dtype.__name__, error)
    print(f"{dtype.__name__}: 16000 samples, largest difference from the analytic tone {error:.1e}")

## Example 3 — every quality level and every sample type

In [ ]:
import numpy as np
import soxr

rng = np.random.default_rng(0)
noise = rng.standard_normal(48000).astype(np.float32) * 0.1
for quality in ("QQ", "LQ", "MQ", "HQ", "VHQ", soxr.VHQ):
    out = soxr.resample(noise, 48000, 16000, quality=quality)
    assert out.shape == (16000,) and np.isfinite(out).all(), (quality, out.shape)
for dtype in (np.float32, np.float64, np.int16, np.int32):
    scale = 1.0 if np.issubdtype(dtype, np.floating) else np.iinfo(dtype).max
    out = soxr.resample((noise * scale).astype(dtype), 48000, 24000)
    assert out.dtype == dtype and out.shape == (24000,), (dtype.__name__, out.dtype, out.shape)
print("5 quality levels (by name and by constant) and float32 / float64 / int16 / int32 all resample")

## Example 4 — streaming in chunks gives the same audio as one call

In [ ]:
import numpy as np
import soxr

in_rate, out_rate = 44100, 16000
signal = np.sin(2 * np.pi * 440.0 * np.arange(in_rate) / in_rate).astype(np.float32)
stream = soxr.ResampleStream(in_rate, out_rate, 1, dtype="float32")
chunks = [signal[start:start + 4410] for start in range(0, in_rate, 4410)]
pieces = [stream.resample_chunk(chunk, last=(index == len(chunks) - 1)) for index, chunk in enumerate(chunks)]
streamed = np.concatenate(pieces)
one_call = soxr.resample(signal, in_rate, out_rate)
assert streamed.shape == one_call.shape == (out_rate,), (streamed.shape, one_call.shape)
difference = float(np.max(np.abs(streamed[1000:-1000] - one_call[1000:-1000])))
assert difference < 1e-4, difference
print(f"10 chunks -> {streamed.shape[0]} samples; largest difference from the one-call result {difference:.1e}")

## Example 5 — stereo input, and bad arguments are refused with an error

In [ ]:
import numpy as np
import soxr

stereo = np.stack([np.sin(np.arange(48000) / 20.0), np.cos(np.arange(48000) / 20.0)], axis=1).astype(np.float32)
out = soxr.resample(stereo, 48000, 16000)
assert out.shape == (16000, 2), out.shape
left_only = soxr.resample(np.ascontiguousarray(stereo[:, 0]), 48000, 16000)
assert float(np.max(np.abs(out[:, 0] - left_only))) < 1e-6
refused = []
for label, call in (("unknown quality", lambda: soxr.resample(stereo, 48000, 16000, quality="best")),
                    ("3-D input", lambda: soxr.resample(np.zeros((4, 4, 4), dtype=np.float32), 2, 1)),
                    ("complex samples", lambda: soxr.resample(np.zeros(16, dtype=np.complex64), 2, 1))):
    try:
        call()
    except (ValueError, TypeError) as error:
        refused.append(f"{label}: {type(error).__name__}")
assert len(refused) == 3, refused
print("stereo keeps its two channels;", "; ".join(refused))